# NB29: balance the main tasks, then test extra CYP3A4 TDI data

Three controlled arms use the same 19 prediction heads, eight masked direct-target
interval columns, scaffold splits, initial seeds and 45-epoch schedule (decay ends at
epoch 40). All arms use interval loss on direct CYP1A2/CYP3A4 and MAE on other heads.
The official CYP3A4 **TDI-condition** target remains a separate auxiliary head; it is
never treated as a direct-inhibition label.

| Arm | Loss weighting | Extra nonchallenge TDI labels |
|---|---|---|
| control | NB28-style per-observation weights: main 1, auxiliary 0.30 | No |
| balanced | Equal mean loss per main head; auxiliary group weight 0.30 | No |
| balanced_extra | Same balanced objective | Yes |

The balanced objective is `(mean(main-head losses) + 0.30 * mean(aux-head losses))/1.30`.
Training-fold label counts define fixed inverse-prevalence coefficients. Uniform
sampling is retained; no challenge oversampling is combined with this change.

All arms already receive challenge TDI labels **only for their permitted training rows**.
Extra source compounds matching any challenge train/blind connectivity key are excluded.
Extra targets matching an existing public molecule are merged into that row. Other extra
molecules are appended only to the extra arm's training set. The extra arm therefore
has slightly more optimizer steps at the same epoch budget; row/step counts are recorded.

Checkpoints use the same epoch-wide macro MAE across direct CYP1A2 and CYP3A4. This gives
both endpoints equal weight. NB29 control is a new internal control, not an exact replay
of NB28: it adds a TDI head and improves the common checkpoint-selection metric.

**Run all:** 15 fits (three arms × five folds). NB28 timings suggest approximately
70–90 minutes on the same GPU, plus setup; this is an estimate. Epoch curves and actual
fold times are saved for a more accurate estimate during this run.

**Resume:** each epoch atomically saves full best/last Lightning checkpoints to Drive.
Run again with the same RUN and configuration. Completed folds are skipped; the current
partial epoch can repeat. Keep all `nb29_<RUN>_*` files. Never run the same arm/fold in
two runtimes concurrently. VARIANTS/FOLDS may select disjoint subsets of the fixed plan.
Changing data, source hashes, model/objective code, or settings requires a new RUN.

**Evaluation:** control→balanced tests weighting; balanced→balanced_extra tests external
TDI data. Compare all endpoints, especially CYP1A2/CYP3A4, on aligned out-of-fold and
holdout predictions. Holdout never chooses epochs or weights. Raw ST-RAE is diagnostic,
not a calibrated submission forecast. This notebook creates no submission automatically.

Based on NB28/NB21, with credit to SuperCowPowers Workbench and Scigantic:
https://github.com/Scigantic/open-admet-challenge-cyp

Official TDI data and assay description:
https://huggingface.co/datasets/openadmet/cyp-challenge-train-test
https://github.com/OpenADMET/Octant_CYP_blog_post/blob/main/protocols/cyp_inhibition_assay.md


## Setup

Choose a new RUN only for a different experiment. Keep it unchanged to resume.


In [ ]:
RUN = 'a'
VARIANTS = ['control', 'balanced', 'balanced_extra']
FOLDS = [0, 1, 2, 3, 4]
MAX_EPOCHS, BATCH = 45, 128
D_H, DEPTH = 700, 5
AUX_WEIGHT = 0.30
BALANCED_AUX_RATIO = 0.30
DECAY_END = 40
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'
import os, tarfile, re
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN)
assert VARIANTS and len(set(VARIANTS)) == len(VARIANTS)
assert set(VARIANTS) <= {'control', 'balanced', 'balanced_extra'}
assert FOLDS and len(set(FOLDS)) == len(FOLDS) and set(FOLDS) <= set(range(5))
assert MAX_EPOCHS >= 45, 'Production run must reach the low-rate phase.'
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/nb29_cache', exist_ok=True)
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/nb29_cache', filter='data')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy', 'scaffold_groups.npy']
assert all(os.path.exists('/content/nb29_cache/' + f) for f in NEED)
print('NB29', RUN, VARIANTS, FOLDS, '| up to', len(VARIANTS) * len(FOLDS), 'fits')


In [ ]:
!pip -q install chemprop==2.3.1 lightning==2.6.6


In [ ]:
BUILD_CODE_SHA256 = '4ef2fc72771f1408bbb3d3df53487d19007c3e8bf9923b437b504e40230b5f63'
import gc, hashlib, json, os, shutil, time, urllib.request, uuid, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/nb29_cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']; PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
GPU=torch.cuda.is_available()
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()
assert len(meta)==len(hold)==len(G)

def cached(name, url, need_cols):
    """Download once to Drive under this notebook's own name. Written to a temporary
    file and renamed, so a half-written copy is never read, by this run or another."""
    p = OUT / name
    def ok(q):
        try:
            return set(need_cols) <= set(pd.read_csv(q, nrows=0).columns)
        except Exception:
            return False
    if not (p.exists() and ok(p)):
        print('downloading', name)
        tmp = Path('/content') / (name + '.dl')
        urllib.request.urlretrieve(url, str(tmp))
        assert ok(tmp), f'{name}: download lacks {need_cols}'
        part = OUT / f'{name}.{uuid.uuid4().hex}.tmp'
        shutil.copyfile(tmp, part); os.replace(part, p)
    return p

# Reuse finished download caches through private copies; never alter older experiments.
for new, old_options in [('aid1851_nb29.csv',['aid1851_nb28.csv','aid1851_nb21.csv','aid1851_nb27.csv','aid1851_nb25.csv']),
                         ('chembl_nb29.csv',['chembl_nb28.csv','chembl_nb21.csv','chembl_nb27.csv','chembl_nb25.csv'])]:
    if not (OUT/new).exists():
        for old in old_options:
            if (OUT/old).exists():
                tmp=OUT/(new+'.'+uuid.uuid4().hex+'.tmp')
                shutil.copyfile(OUT/old,tmp);os.replace(tmp,OUT/new);break
AID = cached('aid1851_nb29.csv',
    'https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
    '&record_type=datatable&actvty=all&response_type=save&aid=1851',
    ['Max_Response', 'Panel Name', 'PUBCHEM_EXT_DATASOURCE_SMILES'])
CHB = cached('chembl_nb29.csv',
    'https://raw.githubusercontent.com/jeremycheminf/openadmet_scripts/main/'
    'CYP_Challenge/data/external/cyp_chembl.csv',
    ['smiles'] + [f'{p}_IC50_PIC50' for p in PAN])

raw=pd.read_csv(AID, low_memory=False, skiprows=[1])
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
# Negated so larger means more inhibition; clipped because a few rows reach +1330.
raw['mr']=(-pd.to_numeric(raw['Max_Response'], errors='coerce')).clip(-50, 150)
raw=raw.dropna(subset=['panel','mr','PUBCHEM_EXT_DATASOURCE_SMILES'])
chembl=pd.read_csv(CHB)
print('AID 1851 rows', len(raw), '| ChEMBL rows', len(chembl))
TDI_URL = 'https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/cyp-challenge-TRAIN_TDI.csv'
TDI_EXPECTED_SHA256 = 'b458f599a792412292664386e8f18adc5d4a4129d6bd212ae80a60fb9b96bb60'
TDI = cached('official_train_tdi_nb29.csv', TDI_URL,
             ['SMILES', 'CYP3A4_pIC50_TDI_condition',
              'CYP3A4_pIC50_TDI_condition_conf_low', 'CYP3A4_pIC50_TDI_condition_conf_high'])
assert hashlib.sha256(TDI.read_bytes()).hexdigest() == TDI_EXPECTED_SHA256, (
    'Official TDI file changed. Do not silently mix source versions; inspect before a new RUN.')


## Data assembly and leakage checks

The preparation code below is embedded so the notebook is self-contained. It aligns
challenge TDI targets by molecular connectivity and excludes every challenge train/blind
key from the external block. Only the ordinary training subset of challenge rows is used
for any head in a fold. Official TDI source bytes are pinned by SHA-256.


In [ ]:
"""Prepare the official CYP3A4 TDI auxiliary target without direct-target pooling.

Exact molecular-connectivity overlaps with every challenge train/blind compound are
excluded from the external block. Challenge targets stay row-aligned so the caller
can apply its ordinary scaffold folds and inner-validation exclusions to every head.
"""

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem


TARGET = "CYP3A4_pIC50_TDI_condition"
LOW = TARGET + "_conf_low"
HIGH = TARGET + "_conf_high"
SOURCES = [
    "https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/blob/main/README.md",
    "https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/cyp-challenge-TRAIN_TDI.csv",
    "https://openadmet.ghost.io/announcing-openadmets-cyp-inhibition-blind-challenge/",
    "https://github.com/OpenADMET/Octant_CYP_blog_post/blob/main/protocols/cyp_inhibition_assay.md",
]


def key_and_smiles(smiles):
    """Match NB28: largest fragment, InChI connectivity block, canonical SMILES."""
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    if mol is None:
        return None, None
    mol = max(Chem.GetMolFrags(mol, asMols=True), key=lambda frag: frag.GetNumHeavyAtoms())
    ikey = Chem.MolToInchiKey(mol)
    return (ikey.split("-")[0] if ikey else None), Chem.MolToSmiles(mol)


def _frame_and_hash(source):
    if isinstance(source, pd.DataFrame):
        frame = source.copy()
        content = frame.to_csv(index=False, lineterminator="\n").encode()
        kind = "canonical_dataframe_csv"
    else:
        path = Path(source)
        content = path.read_bytes()
        frame = pd.read_csv(path)
        kind = "file_bytes"
    return frame, {"sha256": hashlib.sha256(content).hexdigest(), "hash_kind": kind}


def _keyed(frame):
    out = frame.copy()
    cache = {s: key_and_smiles(s) for s in pd.unique(out["SMILES"])}
    pairs = [cache[s] for s in out["SMILES"]]
    out["_key"] = [p[0] for p in pairs]
    out["_smiles"] = [p[1] for p in pairs]
    return out


def prepare_tdi(meta_train, meta_test, tdi_source):
    """Return challenge/extra y, lo, hi, SMILES and an auditable manifest.

    ``meta_train`` and ``meta_test`` accept DataFrames or CSV paths; ``tdi_source``
    accepts the official TRAIN_TDI CSV path or DataFrame. Returned challenge arrays
    have exactly the metadata row order. No labels from external challenge-matching
    keys are ever placed in the extra block. The caller MUST keep challenge holdout,
    outer validation and inner validation rows out of auxiliary-target training too.

    Duplicated connectivity keys get median pIC50 and the union of valid confidence
    intervals. This avoids counting salts/aliases twice or overstating precision.
    A finite label with absent bounds receives a collapsed [y,y] interval and is
    counted in the manifest; reversed or non-enclosing supplied bounds are rejected.
    """
    train, train_hash = _frame_and_hash(meta_train)
    test, test_hash = _frame_and_hash(meta_test)
    source, source_hash = _frame_and_hash(tdi_source)
    for frame in (train, test, source):
        if "SMILES" not in frame:
            raise ValueError("Every input requires a SMILES column")
    if not {TARGET, LOW, HIGH} <= set(source):
        raise ValueError("Official TDI source lacks the target or confidence bounds")

    train, test, source = map(_keyed, (train, test, source))
    if train["_key"].isna().any() or test["_key"].isna().any():
        raise ValueError("A challenge molecule could not be assigned a connectivity key")

    def all_keys(frame):
        keys = set(frame["_key"].dropna())
        if "conn_key" in frame:
            keys |= {str(k).split("-")[0] for k in frame["conn_key"].dropna()}
        return keys

    train_keys, test_keys = all_keys(train), all_keys(test)
    protected = train_keys | test_keys
    if train_keys & test_keys:
        raise ValueError("Train/blind connectivity overlap requires explicit resolution")

    for col in (TARGET, LOW, HIGH):
        source[col] = pd.to_numeric(source[col], errors="coerce")
    finite = np.isfinite(source[TARGET])
    have_bounds = np.isfinite(source[LOW]) & np.isfinite(source[HIGH])
    invalid_bounds = finite & have_bounds & (
        (source[LOW] > source[HIGH])
        | (source[TARGET] < source[LOW])
        | (source[TARGET] > source[HIGH])
    )
    if invalid_bounds.any():
        raise ValueError(f"{int(invalid_bounds.sum())} supplied intervals are invalid")
    missing_bounds = finite & ~have_bounds
    source.loc[missing_bounds, LOW] = source.loc[missing_bounds, TARGET]
    source.loc[missing_bounds, HIGH] = source.loc[missing_bounds, TARGET]

    invalid_smiles = int(source["_key"].isna().sum())
    if "Molecule_Name" in source:
        aliases = source.dropna(subset=["_key"]).groupby("Molecule_Name")["_key"].nunique()
        if (aliases > 1).any():
            raise ValueError("One source Molecule_Name refers to different connectivity keys")

    valid = source.loc[finite & source["_key"].notna()].copy()
    duplicate_groups = []
    records = []
    for key, group in valid.groupby("_key", sort=True):
        if len(group) > 1:
            duplicate_groups.append({
                "key": key,
                "rows": len(group),
                "pIC50_range": float(group[TARGET].max() - group[TARGET].min()),
                "names": sorted(group["Molecule_Name"].dropna().unique().tolist())
                if "Molecule_Name" in group else [],
            })
        records.append({
            "key": key, "SMILES": sorted(group["_smiles"].unique())[0],
            "y": float(group[TARGET].median()),
            "lo": float(group[LOW].min()), "hi": float(group[HIGH].max()),
            "source_rows": len(group),
        })
    collapsed = pd.DataFrame(records).set_index("key", drop=False)
    if not len(collapsed):
        raise ValueError("No finite source TDI labels")

    # Never fall back to Molecule_Name: molecular connectivity is the merge identity.
    challenge = collapsed.reindex(train["_key"].tolist())
    extra = collapsed.loc[~collapsed.index.isin(protected)].copy().reset_index(drop=True)
    if set(extra["key"]) & protected or extra["key"].duplicated().any():
        raise AssertionError("External connectivity deduplication failed")
    if not np.isfinite(extra[["y", "lo", "hi"]].to_numpy()).all():
        raise AssertionError("Nonfinite external targets")
    challenge_y = challenge["y"].to_numpy(float)
    prior = pd.to_numeric(train[TARGET], errors="coerce").to_numpy(float) if TARGET in train else np.full(len(train), np.nan)
    both = np.isfinite(challenge_y) & np.isfinite(prior)
    manifest = {
        "schema": 1,
        "target": TARGET,
        "role": "auxiliary TDI target; never pooled into the direct-inhibition head",
        "key_definition": "largest fragment; InChIKey first block; same as NB28",
        "merge_policy": "connectivity key, not Molecule_Name",
        "dedup_policy": "median label and interval union per connectivity key",
        "sources": SOURCES,
        "input_hashes": {"train": train_hash, "blind": test_hash, "official_tdi": source_hash},
        "source_rows": len(source),
        "source_unique_valid_keys": int(source["_key"].nunique()),
        "source_invalid_smiles": invalid_smiles,
        "source_finite_3a4_labels": int(finite.sum()),
        "source_labels_with_missing_bounds": int(missing_bounds.sum()),
        "source_duplicate_finite_label_groups": duplicate_groups,
        "source_all_rows_outside_challenge": int((~source["_key"].isin(protected) & source["_key"].notna()).sum()),
        "source_blind_matching_rows_excluded": int(source["_key"].isin(test_keys).sum()),
        "challenge_rows": len(train),
        "challenge_tdi_labels_after_connectivity_merge": int(np.isfinite(challenge_y).sum()),
        "challenge_tdi_labels_filled_from_alias": int((np.isfinite(challenge_y) & ~np.isfinite(prior)).sum()),
        "challenge_tdi_labels_changed_by_dedup": int((np.abs(challenge_y[both] - prior[both]) > 1e-9).sum()),
        "challenge_tdi_max_change_from_metadata": float(np.max(np.abs(challenge_y[both] - prior[both]))) if both.any() else None,
        "extra_unique_3a4_labeled_compounds": len(extra),
        "extra_vs_train_or_blind_overlap": 0,
        "extra_pIC50_mean": float(extra["y"].mean()),
        "extra_pIC50_sd": float(extra["y"].std()),
        "extra_pIC50_below4": int((extra["y"] < 4).sum()),
        "extra_table_sha256": hashlib.sha256(extra.to_csv(index=False, lineterminator="\n").encode()).hexdigest(),
        "excluded_sources": ["Octant inhibition release (different assay arm and mostly redundant with official TDI)"],
    }
    return {
        "challenge_keys": train["_key"].to_numpy(),
        "challenge_y": challenge_y,
        "challenge_lo": challenge["lo"].to_numpy(float),
        "challenge_hi": challenge["hi"].to_numpy(float),
        "extra": extra,
        "extra_smiles": extra["SMILES"].to_numpy(),
        "extra_keys": extra["key"].to_numpy(),
        "extra_y": extra["y"].to_numpy(float),
        "extra_lo": extra["lo"].to_numpy(float),
        "extra_hi": extra["hi"].to_numpy(float),
        "manifest": manifest,
    }



def key_and_smiles(s):
    m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
    if m is None:
        return None, None
    frags = Chem.GetMolFrags(m, asMols=True)
    m = max(frags, key=lambda f: f.GetNumHeavyAtoms())
    k = Chem.MolToInchiKey(m)
    return (k.split('-')[0] if k else None), Chem.MolToSmiles(m)

ours = set()
for s in np.concatenate([SM_TR, SM_TE]):
    k, _ = key_and_smiles(s)
    if k: ours.add(k)
for col in ('conn_key',):
    for frame in (meta, mete):
        if col in frame:
            ours |= {str(v).split('-')[0] for v in frame[col].dropna()}
print('challenge connectivity keys', len(ours))

def keyed(smiles):
    ks = {s: key_and_smiles(s) for s in pd.unique(smiles)}
    return [ks[s][0] for s in smiles], [ks[s][1] for s in smiles]

raw['key'], raw['smi'] = keyed(raw['PUBCHEM_EXT_DATASOURCE_SMILES'].to_numpy())
chembl['key'], chembl['smi'] = keyed(chembl['smiles'].to_numpy())
raw = raw.dropna(subset=['key']); chembl = chembl.dropna(subset=['key'])
mr = raw.pivot_table(index='key', columns='panel', values='mr', aggfunc='median').reindex(columns=PAN)
ch = chembl.groupby('key')[[f'{p}_IC50_PIC50' for p in PAN]].median()
ch.columns = PAN
pub_keys = sorted((set(mr.index) | set(ch.index)) - ours)
n_drop = len(set(mr.index) | set(ch.index)) - len(pub_keys)
smi_of = dict(zip(raw['key'], raw['smi'])); smi_of.update(zip(chembl['key'], chembl['smi']))
print(f'public compounds {len(pub_keys)} (dropped {n_drop} that match a challenge compound)')


tdi = prepare_tdi(CACHE/'meta_train.csv.gz', CACHE/'meta_test.csv.gz', TDI)
TDI_INDEX = 18
HEADS = ([f'chal_{i}' for i in ISO] + [f'scr_{i}' for i in ISO]
         + [f'ch_{p}' for p in PAN] + [f'mr_{p}' for p in PAN] + ['tdi_CYP3A4'])
NT = len(HEADS)
assert NT == 19
n_ch = len(SM_TR)
n_pub = len(pub_keys)
pub_position = {key: n_ch + j for j, key in enumerate(pub_keys)}
extra = tdi['extra'].sort_values('key').reset_index(drop=True)
append_extra = extra.loc[~extra['key'].isin(pub_position)].reset_index(drop=True)
SM_ALL = np.concatenate([SM_TR,
    np.array([smi_of[k] for k in pub_keys], dtype=object),
    append_extra['SMILES'].to_numpy(dtype=object)])
ALL_KEYS = np.concatenate([tdi['challenge_keys'], np.array(pub_keys, dtype=object),
                           append_extra['key'].to_numpy(dtype=object)])
assert len(set(ALL_KEYS[n_ch:])) == len(ALL_KEYS[n_ch:])
assert not (set(ALL_KEYS[n_ch:]) & ours), 'Public/extra row leaks a challenge key'
append_position = {key: n_ch + n_pub + j for j, key in enumerate(append_extra['key'])}
EXTRA_IDX = np.array([pub_position[k] if k in pub_position else append_position[k]
                     for k in extra['key']], dtype=int)
APPENDED_IDX = np.arange(n_ch + n_pub, len(SM_ALL))
BASE_PUB_IDX = np.arange(n_ch, n_ch + n_pub)
assert len(np.unique(EXTRA_IDX)) == len(extra)

Y = np.full((len(SM_ALL), NT), np.nan)
for j, iso in enumerate(ISO):
    Y[:n_ch, j] = meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4+j] = meta[f'{iso}_log2fc'].to_numpy(float)
Y[n_ch:n_ch+n_pub, 8:13] = ch.reindex(pub_keys).to_numpy(float)
Y[n_ch:n_ch+n_pub, 13:18] = mr.reindex(pub_keys).to_numpy(float)
Y[:n_ch, TDI_INDEX] = tdi['challenge_y']

LO = np.column_stack([meta[f'{i}_pIC50_direct_inhibition_conf_low'].to_numpy(float) for i in ISO])
HI = np.column_stack([meta[f'{i}_pIC50_direct_inhibition_conf_high'].to_numpy(float) for i in ISO])
lab = np.isfinite(Y[:n_ch, :4])
nob = lab & ~(np.isfinite(LO) & np.isfinite(HI))
LO = np.where(nob, Y[:n_ch, :4], LO)
HI = np.where(nob, Y[:n_ch, :4], HI)
LO[~lab] = np.nan
HI[~lab] = np.nan
assert np.all(LO[lab] <= HI[lab]), 'Reversed direct-target bounds'
B = np.full((len(SM_ALL), 8), np.nan)
B[:n_ch, :4], B[:n_ch, 4:] = LO, HI
YA = np.hstack([Y, B])
NAUG = YA.shape[1]
assert NAUG == 27
YA_EXTRA = YA.copy()
YA_EXTRA[EXTRA_IDX, TDI_INDEX] = extra['y'].to_numpy(float)
YA_BY_VARIANT = {'control': YA, 'balanced': YA, 'balanced_extra': YA_EXTRA}
TRAIN_PUBLIC_BY_VARIANT = {'control': BASE_PUB_IDX, 'balanced': BASE_PUB_IDX,
                          'balanced_extra': np.r_[BASE_PUB_IDX, APPENDED_IDX]}
TW = np.r_[np.ones(4), np.full(NT - 4, AUX_WEIGHT), np.zeros(NAUG - NT)]
assert np.isnan(YA[EXTRA_IDX, TDI_INDEX]).all()
assert np.isfinite(YA_EXTRA[EXTRA_IDX, TDI_INDEX]).all()
assert np.array_equal(YA[:, :TDI_INDEX], YA_EXTRA[:, :TDI_INDEX], equal_nan=True)
assert np.array_equal(YA[:, TDI_INDEX+1:], YA_EXTRA[:, TDI_INDEX+1:], equal_nan=True)
TDI_MANIFEST = dict(tdi['manifest'])
TDI_MANIFEST.update({'external_targets_merged_into_existing_public_rows': len(extra)-len(append_extra),
                     'newly_appended_public_rows': len(append_extra),
                     'all_arms_challenge_tdi_labels': int(np.isfinite(Y[:n_ch,TDI_INDEX]).sum()),
                     'tdi_training_loss': 'standardized MAE on separate auxiliary head',
                     'tdi_confidence_intervals': 'audited and retained in source; not used in this MAE head'})
SOURCE_HASHES = {name: hashlib.sha256(Path(path).read_bytes()).hexdigest()
                 for name, path in [('aid1851', AID), ('chembl', CHB), ('official_tdi', TDI)]}
print('Table', YA.shape, '| base public rows', n_pub,
      '| TDI extra targets', len(extra), '| merged', len(extra)-len(append_extra),
      '| appended', len(append_extra))
print('Base labels per head:', dict(zip(HEADS, np.isfinite(Y).sum(0).tolist())))
print('Direct labels missing intervals:', int(nob.sum()))
print('TDI source SHA-256:', SOURCE_HASHES['official_tdi'])


In [ ]:
import logging, warnings
import lightning as L
import torchmetrics
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from lightning.pytorch.plugins.io import TorchCheckpointIO
from sklearn.preprocessing import StandardScaler
from chemprop import data, featurizers, models, nn
from chemprop.nn.metrics import ChempropMetric
for name in ('lightning', 'lightning.pytorch', 'lightning.fabric'):
    logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings('ignore', module='lightning')

ALL_VARIANTS = ['control', 'balanced', 'balanced_extra']
METRIC_HEADS = [0, 1, 2, 3, TDI_INDEX]
METRIC_NAMES = ISO + ['CYP3A4_TDI']
OBJECTIVE_VERSION = 'nb29-global-fold-head-balanced-v1'

def balanced_coefficients(counts, n_train):
    counts = np.asarray(counts, dtype=float)
    assert counts.shape == (NAUG,) and n_train > 0
    coeff = np.zeros(NAUG, dtype=float)
    group_mass = np.r_[np.full(4, 1/(4*(1+BALANCED_AUX_RATIO))),
                      np.full(NT-4, BALANCED_AUX_RATIO/((NT-4)*(1+BALANCED_AUX_RATIO))),
                      np.zeros(NAUG-NT)]
    np.divide(n_train*group_mass, counts, out=coeff, where=counts > 0)
    return coeff

class IntervalObjective(ChempropMetric):
    def __init__(self, mean, scale, task_weights, balanced=False):
        super().__init__(task_weights=task_weights)
        self.register_buffer('mu', torch.as_tensor(mean, dtype=torch.float))
        self.register_buffer('sd', torch.as_tensor(scale, dtype=torch.float))
        self.balanced = bool(balanced)

    def _calc_unreduced_loss(self, preds, targets, *args):
        preds, targets = preds.float(), targets.float()
        cols = [(preds[:, j]-targets[:, j]).abs() for j in range(preds.shape[1])]
        for j in (0, 3):
            lo_j, hi_j = NT+j, NT+4+j
            p = preds[:,j]*self.sd[j]+self.mu[j]
            lo = targets[:,lo_j]*self.sd[lo_j]+self.mu[lo_j]
            hi = targets[:,hi_j]*self.sd[hi_j]+self.mu[hi_j]
            cols[j] = (torch.relu(lo-p)+torch.relu(p-hi))/self.sd[j]
        return torch.stack(cols, dim=1)

    def update(self, preds, targets, mask=None, weights=None, lt_mask=None, gt_mask=None):
        keep = torch.ones_like(targets, dtype=torch.bool) if mask is None else mask.clone()
        keep[:,NT:] = False
        losses = self._calc_unreduced_loss(preds, targets)
        # where explicitly discards inactive elements; 0 * NaN would not do so.
        losses = torch.where(keep, losses, torch.zeros_like(losses))
        row_weights = torch.ones(len(targets), device=targets.device) if weights is None else weights.reshape(-1)
        self.total_loss += (losses*row_weights[:,None]*self.task_weights).sum()
        self.num_samples += len(targets) if self.balanced else keep.sum()

class EpochHeadMAE(torchmetrics.Metric):
    """Accumulate real-unit sums/counts; no average of minibatch averages."""
    full_state_update = False
    def __init__(self, scale):
        super().__init__()
        self.register_buffer('scale', torch.as_tensor(scale, dtype=torch.float)[METRIC_HEADS])
        self.add_state('error_sum', default=torch.zeros(len(METRIC_HEADS)), dist_reduce_fx='sum')
        self.add_state('count', default=torch.zeros(len(METRIC_HEADS)), dist_reduce_fx='sum')
    def update(self, preds, targets, mask):
        preds, targets = preds.float(), targets.float()
        use = mask[:,METRIC_HEADS]
        err = (preds[:,METRIC_HEADS]-targets[:,METRIC_HEADS]).abs()*self.scale
        self.error_sum += torch.where(use, err, torch.zeros_like(err)).sum(0)
        self.count += use.sum(0)
    def compute(self):
        return torch.where(self.count > 0, self.error_sum/self.count.clamp_min(1),
                           torch.full_like(self.error_sum, float('nan')))

def lr_at(epoch, end=40):
    if epoch < 2:
        return 1e-4+(1e-3-1e-4)*epoch/2
    return 1e-3*(.1**min(1., (epoch-2)/(end-2)))

class ScheduledMPNN(models.MPNN):
    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=1e-4)
        if self.trainer.train_dataloader is None:
            _ = self.trainer.estimated_stepping_batches
        steps = int(self.trainer.num_training_batches)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda step: lr_at(step/steps, self.decay_end)/1e-4)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'interval': 'step'}}
    def on_validation_epoch_start(self):
        self.head_mae.reset()
    def validation_step(self, batch, batch_idx=0):
        bmg, V_d, X_d, targets, weights, lt_mask, gt_mask = batch
        mask = targets.isfinite()
        targets = targets.nan_to_num(nan=0.)
        preds = self.predictor.train_step(self.fingerprint(bmg, V_d, X_d))
        self.head_mae.update(preds, targets, mask)
    def on_validation_epoch_end(self):
        values = self.head_mae.compute()
        if not bool(torch.isfinite(values[[0,3]]).all()):
            raise RuntimeError('Inner validation needs finite labels for both focus endpoints')
        self.log('val_focus_mae', values[[0,3]].mean(), on_epoch=True, on_step=False)
        for name, value in zip(METRIC_NAMES, values):
            self.log('val_mae_'+name, value, on_epoch=True, on_step=False)

def make(scaler, variant, counts, n_train):
    balanced = variant != 'control'
    weights = balanced_coefficients(counts, n_train) if balanced else TW
    crit = IntervalObjective(scaler.mean_, scaler.scale_, weights, balanced)
    mp = nn.BondMessagePassing(d_h=D_H, depth=DEPTH, dropout=0.1)
    ffn = nn.RegressionFFN(n_tasks=NAUG, input_dim=D_H, hidden_dim=300, n_layers=2,
        dropout=0.1, criterion=crit, output_transform=nn.UnscaleTransform.from_standard_scaler(scaler))
    model = ScheduledMPNN(mp, nn.NormAggregation(norm=100), ffn, batch_norm=True,
        metrics=[nn.metrics.MAE()], warmup_epochs=2, init_lr=1e-4, max_lr=1e-3, final_lr=1e-4)
    model.head_mae = EpochHeadMAE(scaler.scale_)
    model.decay_end = DECAY_END
    return model

def atomic_npz(path, **kw):
    path = Path(path)
    tmp = Path(str(path)+'.'+uuid.uuid4().hex+'.tmp.npz')
    np.savez_compressed(tmp, **kw)
    os.replace(tmp, path)

def atomic_json(path, obj):
    path = Path(path)
    tmp = Path(str(path)+'.'+uuid.uuid4().hex+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    os.replace(tmp, path)

class AtomicCheckpointIO(TorchCheckpointIO):
    def save_checkpoint(self, checkpoint, path, storage_options=None):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
        try:
            super().save_checkpoint(checkpoint, str(tmp), storage_options=storage_options)
            os.replace(tmp, path)
        finally:
            if tmp.exists():
                tmp.unlink()

class Every(Callback):
    def __init__(self, path):
        self.path = Path(path)
        self.records = []
        self.prior_seconds = 0.0
        self.t0 = None
        self.epoch_t0 = None
    def elapsed(self):
        return self.prior_seconds + (time.time()-self.t0 if self.t0 is not None else 0.)
    def state_dict(self):
        return {'records': self.records, 'elapsed_seconds': self.elapsed()}
    def load_state_dict(self, state_dict):
        self.records = list(state_dict.get('records', []))
        self.prior_seconds = float(state_dict.get('elapsed_seconds', 0.))
    def on_fit_start(self, trainer, pl_module):
        self.t0 = time.time()
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_t0 = time.time()
    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        metrics = trainer.callback_metrics
        row = {'epoch': int(trainer.current_epoch),
               'lr': float(trainer.optimizers[0].param_groups[0]['lr']),
               'epoch_seconds': time.time()-self.epoch_t0,
               'elapsed_seconds': self.elapsed()}
        for name in ['val_focus_mae'] + ['val_mae_'+x for x in METRIC_NAMES]:
            value = metrics.get(name)
            value = float(value.detach().cpu()) if value is not None else float('nan')
            row[name] = value if np.isfinite(value) else None
        assert row['val_focus_mae'] is not None, 'Missing finite checkpoint metric'
        self.records = [r for r in self.records if r['epoch'] < row['epoch']] + [row]
        atomic_json(self.path, {'records': self.records, 'elapsed_seconds': self.elapsed()})
        recent = np.median([r['epoch_seconds'] for r in self.records[-5:]])
        eta = recent * max(0, trainer.max_epochs-row['epoch']-1) / 60
        print(f"epoch {row['epoch']:2d}: macro MAE {row['val_focus_mae']:.4f} | "
              f"1A2 {row['val_mae_CYP1A2']:.4f} 3A4 {row['val_mae_CYP3A4']:.4f} | "
              f"{row['elapsed_seconds']/60:.1f} min elapsed; ~{eta:.1f} min training left", flush=True)
    def on_fit_end(self, trainer, pl_module):
        self.prior_seconds = self.elapsed()
        self.t0 = None
        atomic_json(self.path, {'records': self.records, 'elapsed_seconds': self.elapsed()})

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()
PTS_BY_VARIANT = {variant: [data.MoleculeDatapoint.from_smi(s, targets[i])
                          for i,s in enumerate(SM_ALL)]
                  for variant, targets in YA_BY_VARIANT.items()}
PTS_TE = [data.MoleculeDatapoint.from_smi(s) for s in SM_TE]

def dl(idx, variant='control', shuffle=False, scaler=None, pts=None):
    ds = data.MoleculeDataset(pts if pts is not None else [PTS_BY_VARIANT[variant][i] for i in idx], FZ)
    ds.cache = True
    sc = None
    if isinstance(scaler, str) and scaler == 'fit':
        sc = StandardScaler().fit(ds.Y)
        # A deliberately absent head must not contaminate masked calculations with NaN.
        absent = ~np.isfinite(sc.mean_) | ~np.isfinite(sc.scale_) | (sc.scale_ <= 0)
        sc.mean_[absent] = 0.
        sc.scale_[absent] = 1.
        ds.normalize_targets(sc)
    elif scaler is not None:
        ds.normalize_targets(scaler)
    return data.build_dataloader(ds, batch_size=BATCH, num_workers=0, shuffle=shuffle), sc

def predict(model, loader):
    pr = L.Trainer(accelerator='auto', devices=1, logger=False, enable_progress_bar=False,
        enable_model_summary=False, precision='16-mixed' if GPU else 32)
    return np.concatenate([p.detach().float().cpu().numpy() for p in pr.predict(model, loader)])[:, :4]


## Controlled fits with full Drive resume

All five scaffold folds and every inner split are hashed before training. The run
configuration also records code, source/data hashes, loss coefficients and fold counts.
Each completed epoch writes a timing/validation curve and atomic full checkpoints.
Per-fold predictions are written atomically only after finite-output and alignment checks.
Changing VARIANTS/FOLDS only chooses work from the fixed five-fold plan.


In [ ]:
import math, types, inspect
y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(Y[:n_ch,:4]).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold)
ho_rows = np.flatnonzero(obs_any & hold)
assert len(ho_rows) == int((obs_any & hold).sum())
assert not set(dev_rows) & set(ho_rows)

def fold_file(k, variant):
    return OUT/f'nb29_{RUN}_{variant}_f{k}.npz'

def array_hash(a):
    a = np.asarray(a)
    return hashlib.sha256(str(a.shape).encode()+str(a.dtype).encode()+a.tobytes()).hexdigest()

def function_code_hash():
    # Ignore interactive cell filenames/line numbers, but hash bytecode and constants.
    def stable(value):
        if isinstance(value, types.CodeType):
            return [value.co_code.hex(), [stable(x) for x in value.co_consts],
                    value.co_names, value.co_varnames, value.co_argcount,
                    value.co_kwonlyargcount, value.co_freevars, value.co_cellvars]
        if isinstance(value, (str,int,float,bool,type(None))):
            return value
        if isinstance(value, (tuple,list)):
            return [stable(x) for x in value]
        return repr(value)
    funcs = [balanced_coefficients, lr_at, make, dl, predict, prepare_tdi,
             key_and_smiles, _frame_and_hash, _keyed, atomic_npz, atomic_json]
    for cls in (IntervalObjective, EpochHeadMAE, ScheduledMPNN, AtomicCheckpointIO, Every):
        funcs.extend(v for v in cls.__dict__.values() if inspect.isfunction(v))
    return hashlib.sha256(json.dumps([stable(f.__code__) for f in funcs], sort_keys=True).encode()).hexdigest()

FOLD_PLAN = []
fold_manifest = []
for k, (tr, va) in enumerate(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows])):
    inner = dev_rows[tr]
    a,b = next(GroupShuffleSplit(1,test_size=.10,random_state=42+k).split(inner,inner,G[inner]))
    train_ch, val, te = inner[a], inner[b], dev_rows[va]
    assert not (set(train_ch) & (set(val)|set(te)|set(ho_rows)))
    assert not set(G[train_ch]) & (set(G[val])|set(G[te])|set(G[ho_rows]))
    plan = {'k': k, 'train_ch': train_ch, 'val': val, 'te': te, 'arms': {}}
    manifest = {'fold': k, 'challenge_train_hash': array_hash(train_ch),
                'inner_val_hash': array_hash(val), 'outer_val_hash': array_hash(te),
                'inner_val_rows': len(val), 'arms': {}}
    for variant in ALL_VARIANTS:
        trn = np.r_[train_ch, TRAIN_PUBLIC_BY_VARIANT[variant]]
        counts = np.isfinite(YA_BY_VARIANT[variant][trn]).sum(0)
        assert (counts[:NT] > 0).all(), 'Every active label head needs training observations'
        coeff = TW if variant == 'control' else balanced_coefficients(counts, len(trn))
        plan['arms'][variant] = {'trn': trn, 'counts': counts, 'coeff': coeff}
        manifest['arms'][variant] = {'train_hash': array_hash(trn), 'train_rows': len(trn),
            'steps_per_epoch': math.ceil(len(trn)/BATCH), 'label_counts': counts.tolist(),
            'loss_coefficients': np.asarray(coeff).tolist()}
    FOLD_PLAN.append(plan)
    fold_manifest.append(manifest)

config = {'schema': 1, 'notebook': 'nb29', 'build_code_sha256': BUILD_CODE_SHA256,
    'runtime_code_sha256': function_code_hash(), 'objective_version': OBJECTIVE_VERSION,
    'epochs': MAX_EPOCHS, 'batch': BATCH, 'd_h': D_H, 'depth': DEPTH,
    'dropout': .1, 'ffn_hidden': 300, 'ffn_layers': 2, 'seed_base': 42,
    'aux_weight_control': AUX_WEIGHT, 'balanced_aux_ratio': BALANCED_AUX_RATIO,
    'sampling': 'uniform; no replacement or challenge oversampling',
    'heads': HEADS, 'task_weights_control': TW.tolist(), 'focus': [0,3],
    'checkpoint_metric': 'epoch macro of separate real-unit CYP1A2 and CYP3A4 MAEs',
    'schedule_end': DECAY_END, 'lr_init': 1e-4, 'lr_peak': 1e-3, 'lr_final': 1e-4,
    'canonical_folds': [0,1,2,3,4], 'all_variants': ALL_VARIANTS, 'fold_plan': fold_manifest,
    'source_hashes': SOURCE_HASHES, 'tdi_manifest': TDI_MANIFEST,
    'data_hashes': {v:array_hash(np.nan_to_num(x,nan=-123456.)) for v,x in YA_BY_VARIANT.items()},
    'structure_hash': hashlib.sha256('\n'.join(SM_ALL).encode()).hexdigest(),
    'split_hash': hashlib.sha256(hold.tobytes()+repr(G.tolist()).encode()).hexdigest(),
    'holdout_hash': array_hash(ho_rows), 'holdout_rows': len(ho_rows),
    'torch': str(torch.__version__), 'lightning': str(L.__version__)}
cfg = OUT/f'nb29_{RUN}_config.json'
CONFIG_SHA256 = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
if cfg.exists():
    assert json.loads(cfg.read_text()) == config, 'Configuration changed: restore it or choose a new RUN.'
else:
    atomic_json(cfg, config)
source_manifest_path = OUT/f'nb29_{RUN}_tdi_manifest.json'
atomic_json(source_manifest_path, TDI_MANIFEST)

for VARIANT in VARIANTS:
    print('\nARM', VARIANT, '| LR at epochs 2/20/40:', [lr_at(e,DECAY_END) for e in [2,20,40]], flush=True)
    for plan in FOLD_PLAN:
        k = plan['k']
        if k not in FOLDS:
            continue
        dest = fold_file(k, VARIANT)
        if dest.exists():
            with np.load(dest) as z:
                assert str(z['config_sha256']) == CONFIG_SHA256
                assert np.array_equal(z['te'],plan['te']) and np.array_equal(z['ho'],ho_rows)
                assert z['oof'].shape == (len(plan['te']),4)
                assert z['hp'].shape == (len(ho_rows),4) and z['tp'].shape == (len(SM_TE),4)
                assert all(np.isfinite(z[name]).all() for name in ('oof','hp','tp'))
            print('already complete:', dest.name)
            continue
        arm = plan['arms'][VARIANT]
        trn, counts, val = arm['trn'], arm['counts'], plan['val']
        print(f'fold {k}: {len(trn)} train rows; {math.ceil(len(trn)/BATCH)} steps/epoch', flush=True)
        L.seed_everything(42+k, workers=True)
        tl, sc = dl(trn, VARIANT, shuffle=True, scaler='fit')
        vl, _ = dl(val, VARIANT, scaler=sc)
        model = make(sc, VARIANT, counts, len(trn))
        ckdir = OUT/f'nb29_{RUN}_checkpoints'/f'{VARIANT}_f{k}'
        ckdir.mkdir(parents=True,exist_ok=True)
        curve_path = OUT/f'nb29_{RUN}_{VARIANT}_f{k}_curve.json'
        timing = Every(curve_path)
        ck = ModelCheckpoint(dirpath=str(ckdir), monitor='val_focus_mae', mode='min',
            filename='best-{epoch:03d}', save_top_k=1, save_last=True, every_n_epochs=1)
        last = ckdir/'last.ckpt'
        trainer = L.Trainer(accelerator='auto', devices=1, max_epochs=MAX_EPOCHS, logger=False,
            enable_progress_bar=False, enable_model_summary=False,
            precision='16-mixed' if GPU else 32, callbacks=[timing,ck],
            plugins=[AtomicCheckpointIO()])
        t0 = time.time()
        trainer.fit(model, tl, vl, ckpt_path=str(last) if last.exists() else None, weights_only=False)
        assert ck.best_model_path and np.isfinite(float(ck.best_model_score))
        saved = torch.load(ck.best_model_path,map_location='cpu',weights_only=False)
        best = make(sc, VARIANT, counts, len(trn))
        best.load_state_dict(saved['state_dict'])
        best.eval()
        oof = predict(best, dl(plan['te'],VARIANT)[0])
        hp = predict(best, dl(ho_rows,VARIANT)[0])
        tp = predict(best, dl(None,VARIANT,pts=PTS_TE)[0])
        assert all(np.isfinite(p).all() for p in (oof,hp,tp)), 'Nonfinite predictions: fold not saved'
        assert len(timing.records) == MAX_EPOCHS, 'Incomplete epoch history'
        atomic_npz(dest, te=plan['te'], ho=ho_rows, oof=oof, hp=hp, tp=tp,
            best_epoch=int(saved['epoch']), best_val_macro_mae=float(ck.best_model_score),
            epochs=MAX_EPOCHS, config_sha256=CONFIG_SHA256, train_rows=len(trn),
            label_counts=counts, loss_coefficients=arm['coeff'],
            scaler_mean=sc.mean_, scaler_scale=sc.scale_,
            elapsed_train_seconds=timing.elapsed(), elapsed_this_session_seconds=time.time()-t0,
            curves_json=json.dumps(timing.records,allow_nan=False))
        print(f'{VARIANT} fold {k} saved; best epoch {saved["epoch"]}; '
              f'{timing.elapsed()/60:.1f} total training minutes; '
              f'{(time.time()-t0)/60:.1f} min this session including predictions',flush=True)
        del model,best,trainer,saved,tl,vl
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
# Collect complete arms; keep every partial fold and curve available for resume.
all_results = {}
mine = [cfg, source_manifest_path]
def strae(y, p, lo, hi):
    use = np.isfinite(y)&np.isfinite(p)&np.isfinite(lo)&np.isfinite(hi)
    y,p,lo,hi = y[use],p[use],lo[use],hi[use]
    loss = lambda x: np.maximum(lo-x,0)+np.maximum(x-hi,0)
    den = loss(np.full_like(y,y.mean())).sum()
    return float(loss(p).sum()/den) if den > 0 else None
for variant in ALL_VARIANTS:
    paths = [fold_file(k,variant) for k in range(5)]
    mine += [p for p in paths if p.exists()]
    mine += [OUT/f'nb29_{RUN}_{variant}_f{k}_curve.json' for k in range(5)
             if (OUT/f'nb29_{RUN}_{variant}_f{k}_curve.json').exists()]
    if not all(p.exists() for p in paths):
        print(variant,'pending folds',[k for k,p in enumerate(paths) if not p.exists()])
        continue
    oof_all = np.full((n_ch,4),np.nan)
    HP,TP,info,seconds = [],[],[],[]
    covered = []
    for k,path in enumerate(paths):
        with np.load(path) as z:
            assert str(z['config_sha256']) == CONFIG_SHA256
            assert np.array_equal(z['ho'],ho_rows) and np.array_equal(z['te'],FOLD_PLAN[k]['te'])
            assert all(np.isfinite(z[name]).all() for name in ('oof','hp','tp'))
            oof_all[z['te']] = z['oof']
            covered.extend(z['te'].tolist())
            HP.append(z['hp'].copy())
            TP.append(z['tp'].copy())
            info.append(int(z['best_epoch']))
            seconds.append(float(z['elapsed_train_seconds']))
    assert len(covered) == len(set(covered)) and set(covered) == set(dev_rows)
    hp,tp = np.mean(HP,0),np.mean(TP,0)
    rows = []
    for j,iso in enumerate(ISO):
        y = y_all[iso]
        d = dev_rows[np.isfinite(y[dev_rows])]
        h = ho_rows[np.isfinite(y[ho_rows])]
        ph = hp[np.isin(ho_rows,h),j]
        p = OUT/f'nb29_{RUN}_{variant}_{iso}.npz'
        atomic_npz(p,dev=d,ho=h,oof=oof_all[d,j],hold=ph,test=tp[:,j],config_sha256=CONFIG_SHA256)
        mine.append(p)
        lo = meta[f'{iso}_pIC50_direct_inhibition_conf_low'].to_numpy(float)
        hi = meta[f'{iso}_pIC50_direct_inhibition_conf_high'].to_numpy(float)
        row = {'iso':iso,'dev_n':len(d),'holdout_n':len(h),
            'dev_spearman':float(spearmanr(y[d],oof_all[d,j]).statistic),
            'holdout_spearman':float(spearmanr(y[h],ph).statistic),
            'dev_mae':float(np.mean(abs(y[d]-oof_all[d,j]))),
            'holdout_mae':float(np.mean(abs(y[h]-ph))),
            'dev_raw_strae':strae(y[d],oof_all[d,j],lo[d],hi[d]),
            'holdout_raw_strae':strae(y[h],ph,lo[h],hi[h])}
        rows.append(row)
        print(variant,row)
    all_results[variant] = {'best_epochs':info,'fold_train_seconds':seconds,'results':rows}
out = OUT/f'results_nb29_{RUN}.json'
atomic_json(out,all_results)
mine.append(out)
print('Results/curves are in Drive:',out,'; full checkpoints remain in Drive separately.')
if not all_results:
    print('No complete arm yet. Resume training with the same RUN/configuration.')


## Compare with the saved v30 fallback

The reference contains our saved predictions, not blind labels. For each completed arm,
the next cell chooses a small blending weight on development predictions only, then
reports the holdout comparison and a scaffold bootstrap interval. The holdout has been
examined repeatedly in this project, so these are exploratory estimates. The cell saves
diagnostics and creates no submission. Keep v30 until the full result is reviewed.


In [ ]:
V30_REFERENCE_BASE64 = '...'
V30_REFERENCE_SHA256 = '890a37223fcb21c527c168245d17f1367c995bd00bbfae1e07603fd0b3b3d217'
V30_SUBMISSION_SHA256 = 'abafa95383f9ac4e5f14290d4d7a6fc274972529b26d2df2fc92e9aba684eaea'
# Compare completed NB29 arms with the saved v30 baseline. Select on development only.
# The reference archive is embedded below by the notebook builder; it contains our own predictions.
import base64, io
reference_bytes=base64.b64decode(V30_REFERENCE_BASE64)
assert hashlib.sha256(reference_bytes).hexdigest()==V30_REFERENCE_SHA256
refbank=np.load(io.BytesIO(reference_bytes))

def band_error(p,lo,hi):return np.maximum(lo-p,0)+np.maximum(p-hi,0)
def fixed_moments(p,reference):
    assert np.isfinite(p).all() and np.std(p)>1e-10
    return np.mean(reference)+(p-np.mean(p))/np.std(p)*np.std(reference)
def band_score(y,p,lo,hi):
    den=band_error(np.full_like(y,np.mean(y)),lo,hi).sum()
    return float(band_error(p,lo,hi).sum()/den)

def scaffold_ci(y,new,old,lo,hi,groups,seed=29):
    unique=np.unique(groups);members=[np.flatnonzero(groups==g) for g in unique]
    rng=np.random.default_rng(seed);d=[]
    for _ in range(2000):
        at=np.concatenate([members[g] for g in rng.integers(0,len(unique),len(unique))])
        d.append(band_score(y[at],new[at],lo[at],hi[at])-band_score(y[at],old[at],lo[at],hi[at]))
    return np.quantile(d,[.025,.975]).tolist()

benchmark={'baseline':'submission_v30.csv','submission_sha256':V30_SUBMISSION_SHA256,
           'weight_grid':[0,.1,.2,.3,.4,.5],
           'warning':'Exploratory: the project holdout has been reused. CIs condition on model/candidate selection. Screen targets cannot establish dense blind potency.',
           'endpoints':{}}
for iso in ['CYP1A2','CYP3A4']:
    prefix=iso+'_';d=refbank[prefix+'dev'];h=refbank[prefix+'ho']
    col=iso+'_pIC50_direct_inhibition';yt=meta[col].to_numpy(float)
    low=meta[col+'_conf_low'].to_numpy(float);high=meta[col+'_conf_high'].to_numpy(float)
    oldd=refbank[prefix+'oof_ref'];oldh=refbank[prefix+'hold_ref'];oldt=refbank[prefix+'test_ref']
    dev_base=band_score(yt[d],oldd,low[d],high[d]);ho_base=band_score(yt[h],oldh,low[h],high[h])
    trials=[]
    for variant in ['control','balanced','balanced_extra']:
        path=OUT/f'nb29_{RUN}_{variant}_{iso}.npz'
        if not path.exists():continue
        z=np.load(path);assert np.array_equal(d,z['dev']) and np.array_equal(h,z['ho'])
        options=[]
        for weight in benchmark['weight_grid']:
            newd=fixed_moments((1-weight)*refbank[prefix+'oof_raw']+weight*z['oof'],oldd)
            options.append((band_score(yt[d],newd,low[d],high[d]),weight))
        dev_score,weight=min(options)
        newh=fixed_moments((1-weight)*refbank[prefix+'hold_raw']+weight*z['hold'],oldh)
        newt=fixed_moments((1-weight)*refbank[prefix+'test_raw']+weight*z['test'],oldt)
        row={'variant':variant,'weight_chosen_on_dev':weight,'dev_strae':dev_score,'dev_delta':dev_score-dev_base,
             'holdout_strae':band_score(yt[h],newh,low[h],high[h]),'holdout_delta':band_score(yt[h],newh,low[h],high[h])-ho_base,
             'holdout_delta_scaffold_ci':scaffold_ci(yt[h],newh,oldh,low[h],high[h],G[h]),
             'holdout_rho':float(spearmanr(yt[h],newh).statistic),'baseline_holdout_rho':float(spearmanr(yt[h],oldh).statistic),
             'blind_rank_correlation_to_v30':float(spearmanr(oldt,newt).statistic)}
        trials.append(row)
    picked=min(trials,key=lambda r:r['dev_strae']) if trials else None
    benchmark['endpoints'][iso]={'baseline_dev_strae':dev_base,'baseline_holdout_strae':ho_base,'trials':trials,'selected_on_development':picked}
    print('\nV30 COMPARISON',iso, json.dumps(benchmark['endpoints'][iso],indent=2))
atomic_json(OUT/f'nb29_{RUN}_v30_benchmark.json',benchmark)
print('Saved comparison. Keep v30 as fallback. No submission CSV or leaderboard upload is made automatically.')


In [ ]:
# Package only after the v30 comparison has been saved.
import zipfile
benchmark_path = OUT/f'nb29_{RUN}_v30_benchmark.json'
assert benchmark_path.exists(), 'Run the v30 comparison cell before downloading results.'
mine.append(benchmark_path)
zip_path = f'/content/nb29_{RUN}_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(set(mine)):
        z.write(p,p.name)
print('Download includes predictions, metrics, timing curves and the v30 comparison.')
print('Full training checkpoints remain in Drive separately.')
if all_results:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print('Browser download unavailable:',type(e).__name__)
else:
    print('Partial results packaged. Resume training with the same RUN/configuration.')
